In [2]:
import os
import time
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

In [8]:
SELECTED_FEATURE_RANGES = {
    # ── Numerical features ──────────────────────────────────────────────────
    
    "sg":   {"type": "numeric", "min": 1.000,  "max": 1.035,
              "label": "Specific Gravity (sg)",
              "hint":  "e.g. 1.020  |  range: 1.000 – 1.035"},

    "al":   {"type": "numeric", "min": 0.0,    "max": 5.0,
              "label": "Albumin Level (al)",
              "hint":  "Scale 0 – 5  |  0 = absent, 5 = heavy"},

    "bgr":  {"type": "numeric", "min": 50.0,   "max": 490.0,
              "label": "Blood Glucose Random (bgr) [mg/dL]",
              "hint":  "e.g. 121  |  normal fasting: 70 – 100 mg/dL"},

    "sc":   {"type": "numeric", "min": 0.1,    "max": 15.0,
              "label": "Serum Creatinine (sc) [mg/dL]",
              "hint":  "e.g. 1.2  |  normal: 0.6 – 1.2 mg/dL"},

    "pcv":  {"type": "numeric", "min": 15.0,   "max": 55.0,
              "label": "Packed Cell Volume (pcv) [%]",
              "hint":  "e.g. 44   |  normal: 35 – 50 %"},

    "rc":   {"type": "numeric", "min": 1.0,    "max": 8.0,
              "label": "Red Blood Cell Count (rc) [mill/mcL]",
              "hint":  "e.g. 5.1  |  normal: 4.5 – 5.5 mill/mcL"},

    "bu":   {"type": "numeric", "min": 10.0,   "max": 200.0,
              "label": "Blood Urea (bu) [mg/dL]",
              "hint":  "e.g. 25   |  normal: 10 – 50 mg/dL"},

    # ── Binary categorical features (0 or 1) ────────────────────────────────
    "htn":  {"type": "binary",
              "label": "Hypertension History (htn)",
              "hint":  "0 = No hypertension  |  1 = Has hypertension"},


    "rbc_is_missing": {"type": "binary",
              "label": "RBC Data Availability (rbc_is_missing)",
              "hint":  "0 = RBC value was recorded  |  1 = RBC value was missing"},
}

In [11]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Ensure your asset loading variables (model, scaler, SCALER_FEATS, SELECTED_FEATS, 
# and SELECTED_FEATURE_RANGES) from your previous blocks are active in memory.

# 2. GENERATE ALL INPUT FIELDS AT ONCE (UPDATED WITH EXPLICIT MIN/MAX LABELS)
inputs_widgets = {}
widget_list = []

# Title banner
widget_list.append(widgets.HTML("<h2>Chronic Kidney Disease (CKD) Clinical Input Panel</h2><hr>"))

for feat, meta in SELECTED_FEATURE_RANGES.items():
    if meta["type"] == "numeric":
        # Overwrite hint to strictly show the required minimum and maximum bounds
        display_hint = f"Allowed Range: {meta['min']} to {meta['max']}"
        input_widget = widgets.FloatText(value=None, placeholder=f"{meta['min']} - {meta['max']}")
    elif meta["type"] == "binary":
        # Keep clear drop-down hints for categorical selections
        display_hint = meta["hint"]
        input_widget = widgets.Dropdown(options=[("0 (No / Normal / Present)", 0), ("1 (Yes / Abnormal / Missing)", 1)], value=0)
        
    # Build clean label HTML with the explicit minimum and maximum subtext
    label_widget = widgets.HTML(f"<b>{meta['label']}</b><br><small style='color: #d9534f; font-weight: bold;'>{display_hint}</small>")
    
    inputs_widgets[feat] = input_widget
    # Pack elements side-by-side using an HBox configuration
    widget_list.append(widgets.HBox([widgets.Box([label_widget], layout=widgets.Layout(width='45%')), 
                                      widgets.Box([input_widget], layout=widgets.Layout(width='55%'))]))

# Prediction button and output container
predict_btn = widgets.Button(description="Run Diagnostics Pipeline", button_style='primary', layout=widgets.Layout(margin='20px 0 0 0', width='98%'))
output_container = widgets.Output()
widget_list.append(predict_btn)
widget_list.append(output_container)

# Display the unified UI dashboard
ui_panel = widgets.VBox(widget_list, layout=widgets.Layout(padding='10px', border='1px solid #ccc', border_radius='5px', width='650px'))
display(ui_panel)

# 3. CLINICAL ADVISORY GENERATOR
def generate_clinical_output(prediction, probability):
    """Generates a structured, highly professional clinical validation report."""
    if prediction == 1:
        status_color = "#d9534f" # Crimson Red
        status_text = "HIGH RISK INDICATED (POSSIBLE CKD PRESENT)"
        clinical_guidance = (
            "<li><b>Urgent Nephrology Referral:</b> Recommend a formal consultation with a certified board nephrologist for comprehensive diagnostic tracking.</li>"
            "<li><b>Confirmatory Testing:</b> Consider scheduling a repeat serum creatinine, comprehensive metabolic panel (CMP), and an imaging study (Renal Ultrasound) to rule out structural anomalies.</li>"
            "<li><b>Therapeutic Optimization:</b> Review current medication regimens. Focus strictly on blood pressure management (target &lt; 130/80 mmHg using ACEIs or ARBs if indicated) and glycemic control. Avoid nephrotoxic agents (NSAIDs, aminoglycosides).</li>"
            "<li><b>Dietary Modification:</b> Advise lifestyle counseling regarding a renal-friendly diet (controlled sodium, tailored protein, and potassium restriction if serum levels are elevated).</li>"
        )
    else:
        status_color = "#5cb85c" # Clinical Green
        status_text = "LOW RISK INDICATED (NO CKD PATTERN DETECTED)"
        clinical_guidance = (
            "<li><b>Routine Monitoring:</b> Maintain routine clinical follow-up intervals based on standard primary care guidelines. Assess eGFR annually.</li>"
            "<li><b>Preventative Management:</b> Continue ongoing management of foundational metabolic parameters (blood pressure titration, HbA1c tracking).</li>"
            "<li><b>Hydration and Wellness:</b> Reinforce adequate hydration targets and the preservation of low-risk cardiovascular habits. Check for regular active exercise and medication compliance.</li>"
        )

    report_html = f"""
    <div style="border: 2px solid {status_color}; border-radius: 6px; padding: 18px; margin-top: 20px; font-family: Arial, sans-serif; background-color: #fafafa;">
        <h3 style="color: {status_color}; margin-top: 0; border-bottom: 1px solid {status_color}; padding-bottom: 8px;">CLINICAL DECISION SUPPORT SYSTEM (CDSS) REPORT</h3>
        <p style="font-size: 14px; margin: 8px 0;"><b>Diagnostic Classification:</b> <span style="color: {status_color}; font-weight: bold;">{status_text}</span></p>
        <p style="font-size: 14px; margin: 8px 0;"><b>Statistical Model Confidence:</b> {probability:.2%}</p>
        <hr style="border: 0; border-top: 1px solid #ddd; margin: 15px 0;">
        <h4 style="margin: 0 0 10px 0; color: #333;">Evidence-Based Recommendations & Next Steps:</h4>
        <ul style="font-size: 13px; line-height: 1.6; margin: 0; padding-left: 20px; color: #555;">
            {clinical_guidance}
        </ul>
        <p style="font-size: 11px; color: #777; margin-top: 15px; font-style: italic; border-top: 1px dashed #ccc; padding-top: 8px;">
            Disclaimer: This output represents an automated algorithm prediction designed for exploratory screening and clinical decision support. It does not replace independent clinical judgment, laboratory confirmation, or formal diagnosis.
        </p>
    </div>
    """
    return report_html

# 4. ERROR HANDLING, VALIDATION, AND INFERENCE LOGIC
def execute_pipeline(b):
    with output_container:
        clear_output()
        errors = []
        user_data = {}
        
        # Phase A: Strict Range Validation
        for feat, meta in SELECTED_FEATURE_RANGES.items():
            val = inputs_widgets[feat].value
            
            # Check for empty numeric fields
            if val is None or val == "":
                errors.append(f"Missing Input: Please enter a valid number for <b>{meta['label']}</b>.")
                continue
                
            val = float(val)
            
            if meta["type"] == "numeric":
                if not (meta["min"] <= val <= meta["max"]):
                    errors.append(f"Out of Bounds: <b>{meta['label']}</b> must be between {meta['min']} and {meta['max']} (Received: {val}).")
            
            user_data[feat] = val

        # If any validation errors exist, stop execution and alert user
        if errors:
            error_html = "<div style='background-color: #f2dede; border: 1px solid #ebccd1; color: #a94442; padding: 12px; border-radius: 4px; margin-top: 15px;'> "
            error_html += "<h4>⚠️ Input Validation Failed:</h4><ul>"
            for err in errors:
                error_html += f"<li>{err}</li>"
            error_html += "</ul>Please adjust the values above and resubmit.</div>"
            display(HTML(error_html))
            return

        # Phase B: Construct Full Inference Row (Padding unselected features with 0)
        full_row_dict = {col: 0.0 for col in SCALER_FEATS}
        for feat, value in user_data.items():
            if feat in full_row_dict:
                full_row_dict[feat] = value
                
        # Transform array into named pandas DataFrame for safe scaling
        inference_df = pd.DataFrame([full_row_dict], columns=SCALER_FEATS)
        
        # Phase C: Mathematical Scaling
        scaled_array = scaler.transform(inference_df)
        scaled_df = pd.DataFrame(scaled_array, columns=SCALER_FEATS)
        
        # Phase D: Model Feature Extraction
        final_model_input = scaled_df[SELECTED_FEATS]
        
        # Phase E: Model Execution
        prediction = int(model.predict(final_model_input)[0])
        probabilities = model.predict_proba(final_model_input)[0]
        confidence_score = probabilities[prediction]
        
        # Phase F: Display Output
        report_view = generate_clinical_output(prediction, confidence_score)
        display(HTML(report_view))

# Bind the execution logic to the user interactive button
predict_btn.on_click(execute_pipeline)

In [17]:
%%writefile ckd_preview_template.html
<div style="border: 2px solid #5cb85c; border-radius: 6px; padding: 18px; margin-top: 20px; font-family: Arial, sans-serif; background-color: #fafafa;">
    <h3 style="color: #5cb85c; margin-top: 0; border-bottom: 1px solid #5cb85c; padding-bottom: 8px;">CLINICAL DECISION SUPPORT SYSTEM (CDSS) REPORT (PREVIEW)</h3>
    <p style="font-size: 14px; margin: 8px 0;"><b>Diagnostic Classification:</b> <span style="color: #5cb85c; font-weight: bold;">LOW RISK INDICATED (NO CKD PATTERN DETECTED)</span></p>
    <p style="font-size: 14px; margin: 8px 0;"><b>Statistical Model Confidence:</b> 99.99%</p>
    <hr style="border: 0; border-top: 1px solid #ddd; margin: 15px 0;">
    <h4 style="margin: 0 0 10px 0; color: #333;">Evidence-Based Recommendations & Next Steps:</h4>
    <ul style="font-size: 13px; line-height: 1.6; margin: 0; padding-left: 20px; color: #555;">
        <li><b>Routine Monitoring:</b> Maintain routine clinical follow-up intervals based on standard primary care guidelines. Assess eGFR annually.</li>
        <li><b>Preventative Management:</b> Continue ongoing management of foundational metabolic parameters (blood pressure titration, HbA1c tracking).</li>
        <li><b>Hydration and Wellness:</b> Reinforce adequate hydration targets and the preservation of low-risk cardiovascular habits. Check for regular active exercise and medication compliance.</li>
    </ul>
    <p style="font-size: 11px; color: #777; margin-top: 15px; font-style: italic; border-top: 1px dashed #ccc; padding-top: 8px;">
        Disclaimer: This output represents an automated algorithm prediction designed for exploratory screening and clinical decision support. It does not replace independent clinical judgment, laboratory confirmation, or formal diagnosis.
    </p>
</div>


Writing ckd_preview_template.html


In [18]:
import os
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Ensure your asset loading variables (model, scaler, SCALER_FEATS, SELECTED_FEATS, 
# and SELECTED_FEATURE_RANGES) from your previous blocks are active in memory.

# 2. GENERATE ALL INPUT FIELDS AT ONCE
inputs_widgets = {}
widget_list = []

widget_list.append(widgets.HTML("<h2>Chronic Kidney Disease (CKD) Clinical Input Panel</h2><hr>"))

for feat, meta in SELECTED_FEATURE_RANGES.items():
    if meta["type"] == "numeric":
        display_hint = f"Allowed Range: {meta['min']} to {meta['max']}"
        input_widget = widgets.FloatText(value=None, placeholder=f"{meta['min']} - {meta['max']}")
    elif meta["type"] == "binary":
        display_hint = meta["hint"]
        input_widget = widgets.Dropdown(options=[("0 (No / Normal / Present)", 0), ("1 (Yes / Abnormal / Missing)", 1)], value=0)
        
    label_widget = widgets.HTML(f"<b>{meta['label']}</b><br><small style='color: #d9534f; font-weight: bold;'>{display_hint}</small>")
    inputs_widgets[feat] = input_widget
    widget_list.append(widgets.HBox([widgets.Box([label_widget], layout=widgets.Layout(width='45%')), 
                                      widgets.Box([input_widget], layout=widgets.Layout(width='55%'))]))

predict_btn = widgets.Button(description="Run Diagnostics Pipeline", button_style='primary', layout=widgets.Layout(margin='20px 0 0 0', width='98%'))
output_container = widgets.Output()
widget_list.append(predict_btn)
widget_list.append(output_container)

ui_panel = widgets.VBox(widget_list, layout=widgets.Layout(padding='10px', border='1px solid #ccc', border_radius='5px', width='650px'))
display(ui_panel)

# 3. CLINICAL ADVISORY GENERATOR
def generate_clinical_output(prediction, probability):
    """Generates a structured, highly professional clinical validation report."""
    if prediction == 1:
        status_color = "#d9534f" 
        status_text = "HIGH RISK INDICATED (POSSIBLE CKD PRESENT)"
        clinical_guidance = (
            "<li><b>Urgent Nephrology Referral:</b> Recommend a formal consultation with a certified nephrologist.</li>"
            "<li><b>Confirmatory Testing:</b> Consider scheduling a repeat serum creatinine and a Renal Ultrasound.</li>"
            "<li><b>Therapeutic Optimization:</b> Focus strictly on blood pressure management. Avoid nephrotoxic agents.</li>"
            "<li><b>Dietary Modification:</b> Advise lifestyle counseling regarding a renal-friendly diet.</li>"
        )
    else:
        status_color = "#5cb85c" 
        status_text = "LOW RISK INDICATED (NO CKD PATTERN DETECTED)"
        clinical_guidance = (
            "<li><b>Routine Monitoring:</b> Maintain routine clinical follow-up intervals. Assess eGFR annually.</li>"
            "<li><b>Preventative Management:</b> Continue ongoing management of foundational metabolic parameters.</li>"
            "<li><b>Hydration and Wellness:</b> Reinforce adequate hydration targets and low-risk habits.</li>"
        )

    return f"""
    <div style="border: 2px solid {status_color}; border-radius: 6px; padding: 18px; margin-top: 20px; font-family: Arial, sans-serif; background-color: #fafafa;">
        <h3 style="color: {status_color}; margin-top: 0; border-bottom: 1px solid {status_color}; padding-bottom: 8px;">CLINICAL DECISION SUPPORT SYSTEM (CDSS) REPORT</h3>
        <p style="font-size: 14px; margin: 8px 0;"><b>Diagnostic Classification:</b> <span style="color: {status_color}; font-weight: bold;">{status_text}</span></p>
        <p style="font-size: 14px; margin: 8px 0;"><b>Statistical Model Confidence:</b> {probability:.2%}</p>
        <hr style="border: 0; border-top: 1px solid #ddd; margin: 15px 0;">
        <h4 style="margin: 0 0 10px 0; color: #333;">Evidence-Based Recommendations & Next Steps:</h4>
        <ul style="font-size: 13px; line-height: 1.6; margin: 0; padding-left: 20px; color: #555;">{clinical_guidance}</ul>
    </div>
    """

# 4. ERROR HANDLING AND INFERENCE LOGIC
def execute_pipeline(b):
    with output_container:
        clear_output()
        errors = []
        user_data = {}
        
        for feat, meta in SELECTED_FEATURE_RANGES.items():
            val = inputs_widgets[feat].value
            if val is None or val == "":
                errors.append(f"Missing Input: Please enter a valid number for <b>{meta['label']}</b>.")
                continue
                
            val = float(val)
            if meta["type"] == "numeric" and not (meta["min"] <= val <= meta["max"]):
                errors.append(f"Out of Bounds: <b>{meta['label']}</b> must be between {meta['min']} and {meta['max']}.")
            user_data[feat] = val

        if errors:
            error_html = "<div style='background-color: #f2dede; color: #a94442; padding: 12px; border-radius: 4px; margin-top: 15px;'><ul>"
            for err in errors: error_html += f"<li>{err}</li>"
            error_html += "</ul></div>"
            display(HTML(error_html))
            return

        full_row_dict = {col: 0.0 for col in SCALER_FEATS}
        for feat, value in user_data.items():
            if feat in full_row_dict: full_row_dict[feat] = value
                
        inference_df = pd.DataFrame([full_row_dict], columns=SCALER_FEATS)
        scaled_df = pd.DataFrame(scaler.transform(inference_df), columns=SCALER_FEATS)
        
        prediction = int(model.predict(scaled_df[SELECTED_FEATS])[0])
        confidence_score = model.predict_proba(scaled_df[SELECTED_FEATS])[0][prediction]
        
        display(HTML(generate_clinical_output(prediction, confidence_score)))

predict_btn.on_click(execute_pipeline)

# 5. RENDER STATIC PREVIEW TO CAPTURE STATE FOR GITHUB
with output_container:
    clear_output()
    if os.path.exists("ckd_preview_template.html"):
        with open("ckd_preview_template.html", "r") as f:
            display(HTML(f.read()))

In [19]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Ensure your asset loading variables (model, scaler, SCALER_FEATS, SELECTED_FEATS, 
# and SELECTED_FEATURE_RANGES) from your previous blocks are active in memory.

# 1. GENERATE ALL INPUT FIELDS AT ONCE
inputs_widgets = {}
widget_list = []

print("=======================================================")
print("  CHRONIC KIDNEY DISEASE (CKD) CLINICAL INPUT PANEL    ")
print("=======================================================")

for feat, meta in SELECTED_FEATURE_RANGES.items():
    if meta["type"] == "numeric":
        display_hint = f"Allowed Range: {meta['min']} to {meta['max']}"
        input_widget = widgets.FloatText(value=None, placeholder=f"{meta['min']} - {meta['max']}")
    elif meta["type"] == "binary":
        display_hint = meta["hint"]
        input_widget = widgets.Dropdown(options=[("0 (No / Normal)", 0), ("1 (Yes / Abnormal)", 1)], value=0)
        
    # Clean plain-text label formatting using standard layout widths
    label_widget = widgets.HTML(f"<b>{meta['label']}</b><br><small style='color: #d9534f;'>{display_hint}</small>")
    inputs_widgets[feat] = input_widget
    
    widget_list.append(widgets.HBox([widgets.Box([label_widget], layout=widgets.Layout(width='45%')), 
                                      widgets.Box([input_widget], layout=widgets.Layout(width='55%'))]))

predict_btn = widgets.Button(description="Run Diagnostics Pipeline", button_style='primary', layout=widgets.Layout(margin='20px 0 0 0', width='98%'))
output_container = widgets.Output()
widget_list.append(predict_btn)
widget_list.append(output_container)

ui_panel = widgets.VBox(widget_list, layout=widgets.Layout(padding='10px', border='1px solid #ccc', border_radius='5px', width='650px'))
display(ui_panel)

# 2. PLAIN TEXT REPORT GENERATOR
def generate_plain_text_output(prediction, probability):
    """Generates a structured, professional clinical report in plain text format."""
    border = "=" * 70
    sub_border = "-" * 70
    
    if prediction == 1:
        status_text = "HIGH RISK INDICATED (POSSIBLE CKD PRESENT)"
        guidance = (
            "- Urgent Nephrology Referral: Recommend a formal consultation with a certified nephrologist.\n"
            "- Confirmatory Testing: Schedule a repeat serum creatinine and a Renal Ultrasound.\n"
            "- Therapeutic Optimization: Focus strictly on blood pressure management. Avoid NSAIDs.\n"
            "- Dietary Modification: Advise lifestyle counseling regarding a renal-friendly diet."
        )
    else:
        status_text = "LOW RISK INDICATED (NO CKD PATTERN DETECTED)"
        guidance = (
            "- Routine Monitoring: Maintain standard clinical follow-up intervals. Assess eGFR annually.\n"
            "- Preventative Management: Continue ongoing management of foundational metabolic parameters.\n"
            "- Hydration and Wellness: Reinforce adequate hydration targets and low-risk habits."
        )

    report = f"""
{border}
CLINICAL DECISION SUPPORT SYSTEM (CDSS) DIAGNOSTIC REPORT
{border}
Diagnostic Classification  : {status_text}
Model Prediction Confidence: {probability:.2%}
{sub_border}
Evidence-Based Recommendations & Next Steps:
{guidance}
{sub_border}
Disclaimer: This output represents an automated algorithm prediction designed 
for exploratory screening and support. It does not replace clinical judgment.
{border}
"""
    print(report)

# 3. ERROR HANDLING, VALIDATION, AND INFERENCE LOGIC
def execute_pipeline(b):
    with output_container:
        clear_output()
        errors = []
        user_data = {}
        
        for feat, meta in SELECTED_FEATURE_RANGES.items():
            val = inputs_widgets[feat].value
            if val is None or val == "":
                errors.append(f"Missing Input: {meta['label']}")
                continue
                
            val = float(val)
            if meta["type"] == "numeric" and not (meta["min"] <= val <= meta["max"]):
                errors.append(f"Out of Bounds: {meta['label']} must be between {meta['min']} and {meta['max']} (Got: {val})")
            user_data[feat] = val

        if errors:
            print("=======================================================")
            print("⚠️  INPUT VALIDATION FAILED:")
            print("=======================================================")
            for err in errors:
                print(f" * {err}")
            print("\nPlease adjust the out-of-bounds values above and rerun.")
            print("=======================================================")
            return

        # Prepare mathematical matrix structures
        full_row_dict = {col: 0.0 for col in SCALER_FEATS}
        for feat, value in user_data.items():
            if feat in full_row_dict: 
                full_row_dict[feat] = value
                
        inference_df = pd.DataFrame([full_row_dict], columns=SCALER_FEATS)
        scaled_df = pd.DataFrame(scaler.transform(inference_df), columns=SCALER_FEATS)
        
        # Run prediction
        prediction = int(model.predict(scaled_df[SELECTED_FEATS])[0])
        confidence_score = model.predict_proba(scaled_df[SELECTED_FEATS])[0][prediction]
        
        # Print standard text report directly into the native notebook output block
        generate_plain_text_output(prediction, confidence_score)

predict_btn.on_click(execute_pipeline)

# 4. DEFAULT VISUAL REFRESH (Saves a placeholder state text block for GitHub)
with output_container:
    clear_output()
    generate_plain_text_output(prediction=0, probability=0.9999)


  CHRONIC KIDNEY DISEASE (CKD) CLINICAL INPUT PANEL    


In [ ]:
# Run this cell to instantly populate the UI fields with standard healthy data
normal_sample = {
    "sg": 1.020,
    "al": 0.0,
    "bgr": 90.0,
    "sc": 0.8,
    "pcv": 42.0,
    "rc": 4.8,
    "bu": 25.0,
    "htn": 0,
    "rbc_is_missing": 0
}

# Inject sample data directly into your active UI widgets
for feat, sample_value in normal_sample.items():
    if feat in inputs_widgets:
        inputs_widgets[feat].value = sample_value

print("✅ UI panel populated with a healthy patient profile. Click 'Run Diagnostics Pipeline' to see the report.")
